In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "genre"
v_esquema = "movie_silver"
v_tabla = "genres"
v_partition = "file_date"
dbutils.widgets.text("p_esquema", v_esquema)
dbutils.widgets.text("p_tabla", v_tabla)

In [0]:
# 1. leemos el archivo csv

genre_schema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("genreName", StringType(), True)
] )

genre_df = spark.read\
    .option("header", True)\
    .schema(genre_schema)\
    .csv(f"{bronze_folder_path}/{v_file_date}/{v_archivo}.csv")

genre_df.printSchema()


In [0]:
# Paso 3 - Renombrar Columnas

genre_renamed_df = genre_df\
    .withColumnRenamed("genreId", "genre_id")\
    .withColumnRenamed("genreName", "genre_name")

display(genre_renamed_df)


In [0]:
# Paso 4 - Añadir columnas a una tabla
genre_final_df = add_ingestion_date(genre_renamed_df)
genre_final_df = add_env(genre_final_df)
genre_final_df = add_file_date(genre_final_df)

display(genre_final_df)
genre_final_df.printSchema()


In [0]:
# Paso 5 - Guardar datos en datalake en formato parket
genre_final_df.write.mode("overwrite").format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
print(f"Se insertaron {genre_final_df.count()} registros en la tabla {v_esquema}.{v_tabla}")


In [0]:
dbutils.notebook.exit("El notebook 03. Ingestion File genre, termino correctamente")